# Stage 4, tier 2: BioLangFusion Phase B, one encoder swapped at a time

**The question.** Does replacing one pretrained encoder improve BioLangFusion while its splits, 999-nucleotide rule, codon grid, fusion heads, TextCNN and training recipe stay fixed? Phase A established the original three-encoder comparison. Phase B tests the replacements below and asks whether the complementarity measured in Stage 1 helps explain which swaps benefit fusion. This is research plan item 1, Phase B ([Gantt task 24](../../docs/Research_Project_Plan_Gantt.xlsx); Overleaf Section 12.2, item 1).

**Compute device and cost.** The notebook supports NVIDIA GPUs through CUDA, Apple GPUs through MPS, and CPU execution. Setup reports the selected device and GPU name before model loading, embedding extraction or training. No hardware-specific runtime or peak RAM/VRAM requirement is established here. The storage projection below estimates token-array disk space; it is not a memory requirement or a duration forecast.

**What changes.** Each comparison replaces one encoder in its own modality's slot:

| Slot | Original encoder | Replacement | Placement on the codon grid |
| --- | --- | --- | --- |
| DNA | Nucleotide Transformer v2 100M | NT 500M human-ref | 6-mers, using the same transposed convolution |
| DNA | Nucleotide Transformer v2 100M | HyenaDNA large | Nucleotide tokens averaged in threes |
| RNA | RNA-FM | RiNALMo | Nucleotide tokens averaged in threes |
| RNA | RNA-FM | mRNA-FM, CaLM | Codon tokens used directly |
| Protein | ESM-2 8M | ESM-2 35M, ESM-2 150M, ProtBERT | Residue tokens used directly |

**Why these comparisons.** The plan names CaLM, mRNA-FM, ProtBERT, ESM-2 150M and alternative DNA encoders. RiNALMo and ESM-2 35M extend the comparison to encoders that Stage 1 found similar to those they replace. Under H2, that similarity motivates testing whether they add less useful information; it does not establish the outcome. DNABERT-2 remains outside this sweep because its variable-length byte-pair tokens need a separately designed, fixed grid adapter.

---

## Setup and prerequisites

### Imports and compute device

Import the shared data, encoder, token-cache and training functions, then inspect the reported device before proceeding. Device availability depends on the installed PyTorch build and local hardware. The cache setting `DTYPE = np.float16` below controls stored embedding arrays; it does not by itself select half-precision encoder loading or training.

In [1]:
# Standard library
import datetime
import json
import os
import time
from pathlib import Path

# Run any operation the Apple GPU backend lacks on the CPU; must precede importing torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Data handling
import numpy as np
import pandas as pd
import torch

# Shared project code in Code/shared_code; this notebook lives in Code/stage4
import sys
CODE_DIR = next((candidate.resolve()
                 for parent in (Path.cwd(), *Path.cwd().parents)
                 for candidate in (parent, parent / "Code")
                 if (candidate / "shared_code").is_dir()), None)
if CODE_DIR is None:
    raise RuntimeError("Open this repository in VS Code before running this notebook.")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import datasets, encoders
from shared_code.encoders import ENCODERS
from shared_code.fusion import blf, train
from shared_code.fusion import token_embeddings as tokens

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| Apple MPS available:", torch.backends.mps.is_available())
if DEVICE == "cuda":
    print(f"Selected device: {DEVICE} | GPU: {torch.cuda.get_device_name()}")
elif DEVICE == "mps":
    print(f"Selected device: {DEVICE} | GPU: Apple GPU (MPS)")
else:
    print(f"Selected device: {DEVICE}")

torch 2.10.0+cu126 | CUDA available: True | Apple MPS available: False
Selected device: cuda | GPU: NVIDIA GeForce RTX 2060 SUPER


### Settings and comparison scope

**Benchmarks.** The default sweep covers CoV-Vac, fungal expression, E. coli and mRNA stability at seed 0. Comparisons use the same test rows as the original-encoder runs. Ab1 is named in the plan but is not included because its data are not public.

**The stability subset.** Stability has 23,929 rows, compared with 3,138 for fungal, so the default limits it to mRNA-FM, CaLM and ESM-2 150M. Row count alone does not determine runtime or peak memory. The [October 19 mRFP control record](../../docs/run-records.md#2026-10-19-stage4-mrfp-control-the-mrfp-synonymous-control-research-plan-item-5) motivates mRNA-FM: on that reporter dataset it added 0.182 held-out expression $R^2$ above the selected composition and structure controls, averaged over ten grouped-fold assignments. This motivates a transfer test on stability; it does not show that the same gain will transfer. CaLM supplies another codon-resolution RNA encoder, while ESM-2 150M tests a larger protein encoder.

**What stays fixed.** Training retains Phase A's Equation 1 convolution, 100 TextCNN channels and recipe. The gated head uses the entropy weight selected by validation for the original encoders on each benchmark. It is not tuned again for each replacement.

**Where work is saved.** Token caches live under `Code/stage3/embeddings/<benchmark>/<encoder>/`. Phase B results live under `Code/stage4/results/phaseB/<benchmark>/<encoder>/`, including JSON records with per-row test predictions. The cells below first list the selected comparisons and estimate their token-array storage, then check the original results needed for comparison.

In [2]:
BENCHMARKS = ("cov_vac_degradation", "fungal_expression", "ecoli_proteins",
              "mrna_stability")                                 # smallest first, costliest last
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "phaseB"
ORIGINAL_RESULTS = {                                   # the original encoders' runs
    "cov_vac_degradation": [Path("stage4/results/phaseA/cov_vac_degradation")],
    "ecoli_proteins": [Path("stage4/results/phaseA/ecoli_proteins")],
    "fungal_expression": [Path("stage3/results/fungal_expression"),
                          Path("stage4/results/phaseA/fungal_expression")],
    "mrna_stability": [Path("stage4/results/phaseA/mrna_stability")],
}
ORIGINAL_RESULTS = {key: [CODE_DIR / folder for folder in folders]
                    for key, folders in ORIGINAL_RESULTS.items()}
STAGE1_ALIGNMENT = CODE_DIR.parent / "docs" / "figures" / "stability" / "results.json"

TCONV = "equation"
CHANNELS = 100
BATCH_SIZE = 32
MAX_EPOCHS = 500
SEED = 0
CHUNK_SIZE = tokens.DEFAULT_CHUNK
DTYPE = np.float16

SWAPS = [("dna", "nt500m"), ("dna", "hyenadna"),
         ("rna", "rinalmo"), ("rna", "mrnafm"), ("rna", "calm"),
         ("prot", "esm2_35m"), ("prot", "esm2_150m"), ("prot", "protbert")]
# mRNA stability is 23,929 sequences: all eight swaps would be about 100 GB of token cache.
# These three are the mRFP control's prediction and its two controls; widen the list here
# once the disk projection below says the rest fit.
STABILITY_SWAPS = [("rna", "mrnafm"), ("rna", "calm"), ("prot", "esm2_150m")]
ORIGINAL = dict(blf.ENCODER_OF_MODALITY)


def swaps_for(key):
    """The swaps run on one benchmark: all of them, except on the costly one."""
    return STABILITY_SWAPS if key == "mrna_stability" else SWAPS

def required_encoders(key):
    """Original streams and swaps, with each shared cache visited once."""
    return list(dict.fromkeys([*ORIGINAL.values(), *(e for _, e in swaps_for(key))]))


planned = sum(len(swaps_for(k)) for k in BENCHMARKS)
print(f"{planned} swap-benchmark pairs over {len(BENCHMARKS)} benchmarks, device {DEVICE}")
pd.DataFrame([{"slot": m, "replaces": ENCODERS[ORIGINAL[m]].label, "swapped in": ENCODERS[k].label,
               "grid": tokens.grid_resolution(k)} for m, k in SWAPS])

27 swap-benchmark pairs over 4 benchmarks, device cuda


,slot,replaces,swapped in,grid
0,dna,NT v2 100M multi-species,NT 500M human-ref,kmer
1,dna,NT v2 100M multi-species,HyenaDNA large,nucleotide
2,rna,RNA-FM,RiNALMo 150M,nucleotide
3,rna,RNA-FM,mRNA-FM,codon
4,rna,RNA-FM,CaLM,codon
5,prot,ESM-2 8M,ESM-2 35M,codon
6,prot,ESM-2 8M,ESM-2 150M,codon
7,prot,ESM-2 8M,ProtBERT,codon


In [3]:
# What the extraction will cost, before it starts. Tokens per sequence come from each
# encoder's own tokenization rule, so this is arithmetic rather than a guess, and the widths
# are the encoders' hidden sizes. Disk is the binding constraint on mRNA stability.
WIDTHS = {"nt500m": 1280, "hyenadna": 256, "rinalmo": 1280, "mrnafm": 1280, "calm": 768,
          "esm2_35m": 480, "esm2_150m": 640, "protbert": 1024}

projection = []
for key in BENCHMARKS:
    lengths = datasets.blf_benchmark_rows(key)["length"].to_numpy()
    for modality, encoder_key in swaps_for(key):
        per_row = np.array([tokens.tokens_for_length(ENCODERS[encoder_key], int(n))
                            for n in lengths])
        gigabytes = per_row.sum() * WIDTHS[encoder_key] * np.dtype(DTYPE).itemsize / 1e9
        projection.append({"benchmark": key, "encoder": ENCODERS[encoder_key].label,
                           "rows": len(lengths), "GB": round(gigabytes, 1)})
projection = pd.DataFrame(projection)
display(projection.pivot_table(index="benchmark", columns="encoder", values="GB",
                               aggfunc="sum").fillna(""))
print(f"total token cache for the planned swaps: {projection['GB'].sum():.0f} GB")
print("All of it stays out of Git, under Code/stage3/embeddings/<benchmark>/<encoder>/.")

encoder,CaLM,ESM-2 150M,ESM-2 35M,HyenaDNA large,NT 500M human-ref,ProtBERT,RiNALMo 150M,mRNA-FM
benchmark,,,,,,,,
cov_vac_degradation,0.1,0.1,0.1,0.1,0.1,0.1,0.5,0.2
ecoli_proteins,1.3,1.1,0.8,1.3,1.1,1.8,6.7,2.2
fungal_expression,1.0,0.9,0.6,1.0,0.9,1.4,5.1,1.7
mrna_stability,8.0,6.7,,,,,,13.4


total token cache for the planned swaps: 58 GB
All of it stays out of Git, under Code/stage3/embeddings/<benchmark>/<encoder>/.


**Reading the storage projection.** The table above multiplies expected content-token counts by a hard-coded hidden width and the cache dtype size. Its decimal GB values cover token-array payload only, excluding model downloads, archive and metadata overhead, training checkpoints and other saved weights. They do not establish which resource will limit a run.

**Known estimate limitation.** The projection uses width 1,280 for RiNALMo. Earlier extraction outputs inspected before the cache reset reported 640, but those legacy outputs do not verify the current cache. If a verified extraction confirms width 640, the current projection would overestimate its token payload by a factor of two. Check the width against that extraction before relying on this estimate. The older “about 100 GB” comment in Settings is also an unverified approximation. These calculations and comments are preserved; neither should be treated as a verified storage requirement.

### Check the original encoders' results

Phase B needs these runs both as score references and to select the gated head's entropy weight: Phase A for CoV-Vac, E. coli and mRNA stability; Stage 3 step 3 or the configured Phase A folder for fungal. Check the table below for available variants and a gated lambda before starting training. The configured paths identify where those prerequisites are expected; they do not generate missing results.

In [4]:
def label(r):
    return "mil (lam 0)" if r.variant == "mil" and r.config["entropy_lambda"] == 0 else r.variant


def original_runs(key):
    """Seed-0 runs of BioLangFusion's own encoders at Phase A's settings.

    A unimodal model reads no grid, so its runs count under either convolution setting.
    """
    runs = []
    for folder in ORIGINAL_RESULTS[key]:
        for path in sorted(folder.glob("*.json")):
            r = train.Result.load(path)
            if r.config["channels"] != CHANNELS or r.config["seed"] != SEED:
                continue
            if r.variant.startswith("unimodal:") or r.config.get("tconv") == TCONV:
                runs.append(r)
    return runs


def original_lambda(key):
    gated = [r for r in original_runs(key) if r.variant == "mil"
             and r.config["entropy_lambda"] > 0 and r.val_score is not None]
    if not gated:
        raise RuntimeError(f"{key}: no gated-head runs for the original encoders yet; "
                           "run Phase A first")
    return max(gated, key=lambda r: r.val_score).config["entropy_lambda"]


status = []
for key in BENCHMARKS:
    runs = original_runs(key)
    have = sorted({label(r) for r in runs})
    try:
        lam = original_lambda(key)
    except RuntimeError:
        lam = None
    status.append({"benchmark": key, "original runs": len(runs), "variants": ", ".join(have),
                   "gated lambda": lam})
pd.DataFrame(status).set_index("benchmark")

,original runs,variants,gated lambda
benchmark,,,
cov_vac_degradation,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01
fungal_expression,11,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01
ecoli_proteins,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01
mrna_stability,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01


---

## Part 1: Prepare the token embeddings

### Check caches before loading models

Inspect every required cache, including the original three streams. The preflight checks extraction settings, recorded ordered-input hashes, shard structure, consistent widths and each row's token count. Available historical shard hashes are also checked. Verification can be reused between preflight and extraction while file statistics remain unchanged; rerunning this preflight creates a fresh verification object.

**Input identity is required.** Reusing a cache requires recorded ordered-input identity; this notebook has no legacy-cache reuse opt-in. New extraction records that identity before the first shard, and a mismatch stops reuse. The [reset record](../../docs/run-records.md#2026-10-25-encoder-swap-unverified-cache-reset) preserves the history of the legacy-cache cleanup.


In [5]:
recovery_path = CACHE_ROOT / "recovery-records.json"
RECOVERY_RECORDS = ({r["key"]: r for r in json.loads(recovery_path.read_text())["caches"]}
                    if recovery_path.exists() else {})
CACHE_VERIFICATION = tokens.CacheVerification()
CACHE_PLANS = {}
checks = []
for benchmark in BENCHMARKS:
    rows = datasets.blf_benchmark_rows(benchmark)
    for encoder_key in required_encoders(benchmark):
        encoder = ENCODERS[encoder_key]
        texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
        plan = tokens.inspect_cache(
            encoder, texts, CACHE_ROOT / benchmark / encoder_key,
            chunk_size=CHUNK_SIZE, dtype=DTYPE,
            recovery_evidence=RECOVERY_RECORDS.get(f"{benchmark}/{encoder_key}"),
            verification=CACHE_VERIFICATION)
        CACHE_PLANS[benchmark, encoder_key] = plan
        checks.append({"benchmark": benchmark, "encoder": encoder.label,
                       "status": plan.status, "reusable shards": len(plan.reusable_shards),
                       "missing shards": len(plan.missing_shards), "input provenance": plan.provenance,
                       "issues": "; ".join(plan.issues)})
        print(f"  {benchmark}/{encoder_key}: {plan.status}, {plan.provenance}", flush=True)
display(pd.DataFrame(checks).set_index(["benchmark", "encoder"]))
blocked = [plan.summary for plan in CACHE_PLANS.values() if not plan.ready]
if blocked:
    raise tokens.CacheRecoveryRequired("Cache preflight stopped before model loading:\n" + "\n".join(blocked))
missing = sum(len(plan.missing_shards) for plan in CACHE_PLANS.values())
print(f"Preflight passed: {len(CACHE_PLANS)} caches; {missing} shards need extraction.")


  cov_vac_degradation/ntv2_100m: complete, recorded_input
  cov_vac_degradation/rnafm: complete, recorded_input
  cov_vac_degradation/esm2_8m: complete, recorded_input
  cov_vac_degradation/nt500m: complete, recorded_input
  cov_vac_degradation/hyenadna: complete, recorded_input
  cov_vac_degradation/rinalmo: complete, recorded_input
  cov_vac_degradation/mrnafm: complete, recorded_input
  cov_vac_degradation/calm: complete, recorded_input
  cov_vac_degradation/esm2_35m: complete, recorded_input
  cov_vac_degradation/esm2_150m: complete, recorded_input
  cov_vac_degradation/protbert: complete, recorded_input
  fungal_expression/ntv2_100m: complete, recorded_input
  fungal_expression/rnafm: complete, recorded_input
  fungal_expression/esm2_8m: complete, recorded_input
  fungal_expression/nt500m: complete, recorded_input
  fungal_expression/hyenadna: complete, recorded_input
  fungal_expression/rinalmo: complete, recorded_input
  fungal_expression/mrnafm: complete, recorded_input
  funga

status  reusable shards  missing shards input provenance issues
benchmark           encoder                                                                                    
cov_vac_degradation NT v2 100M multi-species  complete               10               0   recorded_input       
                    RNA-FM                    complete               10               0   recorded_input       
                    ESM-2 8M                  complete               10               0   recorded_input       
                    NT 500M human-ref         complete               10               0   recorded_input       
                    HyenaDNA large            complete               10               0   recorded_input       
                    RiNALMo 150M              complete               10               0   recorded_input       
                    mRNA-FM                   complete               10               0   recorded_input       
                    CaLM                      complete               10               0   recorded_input       
                    ESM-2 35M                 complete               10               0   recorded_input       
                    ESM-2 150M                complete               10               0   recorded_input       
                    ProtBERT                  complete               10               0   recorded_input       
fungal_expression   NT v2 100M multi-species  complete               13               0   recorded_input       
                    RNA-FM                    complete               13               0   recorded_input       
                    ESM-2 8M                  complete               13               0   recorded_input       
                    NT 500M human-ref         complete               13               0   recorded_input       
                    HyenaDNA large            complete               13               0   recorded_input       
                    RiNALMo 150M              complete               13               0   recorded_input       
                    mRNA-FM                   complete               13               0   recorded_input       
                    CaLM                      complete               13               0   recorded_input       
                    ESM-2 35M                 complete               13               0   recorded_input       
                    ESM-2 150M                complete               13               0   recorded_input       
                    ProtBERT                  complete               13               0   recorded_input       
ecoli_proteins      NT v2 100M multi-species  complete               18               0   recorded_input       
                    RNA-FM                    complete               18               0   recorded_input       
                    ESM-2 8M                  complete               18               0   recorded_input       
                    NT 500M human-ref         complete               18               0   recorded_input       
                    HyenaDNA large            complete               18               0   recorded_input       
                    RiNALMo 150M              complete               18               0   recorded_input       
                    mRNA-FM                   complete               18               0   recorded_input       
                    CaLM                      complete               18               0   recorded_input       
                    ESM-2 35M                 complete               18               0   recorded_input       
                    ESM-2 150M                complete               18               0   recorded_input       
                    ProtBERT                  complete               18               0   recorded_input       
mrna_stability      NT v2 100M multi-species  complete               94               0   recorded_input       
                

Preflight passed: 39 caches; 0 shards need extraction.


**Reading the cache-status table.** Each row describes one benchmark and encoder cache. A shard is a saved chunk of token embeddings; shard counts describe stored inputs rather than trained models.

| Field | Meaning |
| --- | --- |
| `status` | `complete` means all expected shards are present and the preflight reports no issues. It describes embedding readiness, not completion of downstream training. |
| `reusable shards` | Existing chunks that passed the applicable reuse checks. |
| `missing shards` | Expected chunks that are absent and would need extraction. |
| `input provenance` | `recorded_input` means the cache stores an identity for its ordered input sequences. Successful preflight with no issues establishes that this identity matches the requested inputs. |
| `issues` | Problems that block reuse and need inspection before extraction or training. |

When every required cache is complete, has no missing shards and reports no issues, no new embedding computation is needed. The reuse cell below still opens those caches and reports what it found.

### Extract missing shards and reuse complete caches

Open complete caches and compute only missing chunks permitted by the preflight. The original streams are included, so a missing original cache cannot be overlooked before training. Complete caches can be reused without loading an encoder; missing work may require model downloads and embedding computation.

Extraction records input identity before writing its first shard and checks token counts and finite values before publishing each new shard. Finished shards survive a failure; an unfinished shard may need to be repeated. A printed sequence count is progress, while a published shard is recoverable work. Cache and best-weight files remain local and ignored by Git.

The resulting table distinguishes extraction from reuse. Its GB column reports stored token payload, and the elapsed time describes that particular invocation and its mix of reused and computed work.

In [6]:
started = time.perf_counter()
written = []
for key in BENCHMARKS:
    rows = datasets.blf_benchmark_rows(key)
    print(f"=== {key}: {len(rows)} sequences ({datetime.datetime.now():%H:%M:%S}) ===")
    for encoder_key in required_encoders(key):
        encoder = ENCODERS[encoder_key]
        texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
        cache, computed = tokens.extract(encoder, texts, CACHE_ROOT / key / encoder_key, DEVICE,
                                         chunk_size=CHUNK_SIZE, dtype=DTYPE,
                                         recovery_evidence=RECOVERY_RECORDS.get(f"{key}/{encoder_key}"),
                                         verification=CACHE_VERIFICATION)
        written.append({"benchmark": key, "encoder": encoder.label, "sequences": len(cache),
                        "tokens": cache.meta["total_tokens"], "width": cache.width,
                        "GB": round(cache.meta["bytes"] / 1e9, 2),
                        "this run": "extracted" if computed else "already complete"})
        cache.close()
        print(f"  {encoder.label}: {written[-1]['this run']}, {written[-1]['GB']} GB")
print(f"\nextraction finished in {(time.perf_counter() - started) / 60:.1f} minutes")
pd.DataFrame(written).set_index(["benchmark", "encoder"])

=== cov_vac_degradation: 2400 sequences (13:46:00) ===
  NT v2 100M multi-species: already complete, 0.04 GB
  RNA-FM: already complete, 0.25 GB
  ESM-2 8M: already complete, 0.04 GB
  NT 500M human-ref: already complete, 0.1 GB
  HyenaDNA large: already complete, 0.1 GB
  RiNALMo 150M: already complete, 0.25 GB
  mRNA-FM: already complete, 0.17 GB
  CaLM: already complete, 0.1 GB
  ESM-2 35M: already complete, 0.06 GB
  ESM-2 150M: already complete, 0.08 GB
  ProtBERT: already complete, 0.13 GB
=== fungal_expression: 3138 sequences (13:46:01) ===
  NT v2 100M multi-species: already complete, 0.35 GB
  RNA-FM: already complete, 2.56 GB
  ESM-2 8M: already complete, 0.43 GB
  NT 500M human-ref: already complete, 0.86 GB
  HyenaDNA large: already complete, 1.02 GB
  RiNALMo 150M: already complete, 2.56 GB
  mRNA-FM: already complete, 1.71 GB
  CaLM: already complete, 1.02 GB
  ESM-2 35M: already complete, 0.64 GB
  ESM-2 150M: already complete, 0.85 GB
  ProtBERT: already complete, 1.37 

sequences    tokens  width     GB          this run
benchmark           encoder                                                                      
cov_vac_degradation NT v2 100M multi-species       2400     38400    512   0.04  already complete
                    RNA-FM                         2400    194400    640   0.25  already complete
                    ESM-2 8M                       2400     64800    320   0.04  already complete
                    NT 500M human-ref              2400     38400   1280   0.10  already complete
                    HyenaDNA large                 2400    194400    256   0.10  already complete
                    RiNALMo 150M                   2400    194400    640   0.25  already complete
                    mRNA-FM                        2400     64800   1280   0.17  already complete
                    CaLM                           2400     64800    768   0.10  already complete
                    ESM-2 35M                      2400     64800    480   0.06  already complete
                    ESM-2 150M                     2400     64800    640   0.08  already complete
                    ProtBERT                       2400     64800   1024   0.13  already complete
fungal_expression   NT v2 100M multi-species       3138    337376    512   0.35  already complete
                    RNA-FM                         3138   2000436    640   2.56  already complete
                    ESM-2 8M                       3138    666812    320   0.43  already complete
                    NT 500M human-ref              3138    337376   1280   0.86  already complete
                    HyenaDNA large                 3138   2000436    256   1.02  already complete
                    RiNALMo 150M                   3138   2000436    640   2.56  already complete
                    mRNA-FM                        3138    666812   1280   1.71  already complete
                    CaLM                           3138    666812    768   1.02  already complete
                    ESM-2 35M                      3138    666812    480   0.64  already complete
                    ESM-2 150M                     3138    666812    640   0.85  already complete
                    ProtBERT                       3138    666812   1024   1.37  already complete
ecoli_proteins      NT v2 100M multi-species       4450    442550    512   0.45  already complete
                    RNA-FM                         4450   2622015    640   3.36  already complete
                    ESM-2 8M                       4450    874005    320   0.56  already complete
                    NT 500M human-ref              4450    442550   1280   1.13  already complete
                    HyenaDNA large                 4450   2622015    256   1.34  already complete
                    RiNALMo 150M                   4450   2622015    640   3.36  already complete
                    mRNA-FM                        4450    874005   1280   2.24  already complete
                    CaLM                           4450    874005    768   1.34  already complete
                    ESM-2 35M                      4450    874005    480   0.84  already complete
                    ESM-2 150M                     4450    874005    640   1.12  already complete
                    ProtBERT                       4450    874005   1024   1.79  already complete
mrna_stability      NT v2 100M multi-species      23929   2642703    512   2.71  already complete
                    RNA-FM                        23929  15678663    640  20.07  already complete
                    ESM-2 8M                      23929   5226221    320   3.34  already complete
                    mRNA-FM                       23929   5226221   1280  13.38  already complete
                    CaLM                          23929   5226221    768   8.03  already complete
                    ESM-2 150M                    23929   5226221    640   6.69  already complete

**Reading the embedding inventory.** The table above describes the cached numerical features available to the prediction models.

| Field | Meaning |
| --- | --- |
| `sequences` | Benchmark rows represented in the cache, including repeated sequences if the benchmark contains them. |
| `tokens` | Total represented positions across those rows. Tokenization differs by encoder, so encoders can represent the same sequences with different token counts. |
| `width` | The number of values in each token's embedding vector. |
| `GB` | Stored token-array payload in decimal gigabytes, calculated from token count, width and dtype size. It excludes archive and metadata overhead and is not a peak RAM or GPU-memory requirement. |
| `this run` | `already complete` means this invocation reused the cache without loading the encoder. `extracted` means it computed missing chunks, which may be only part of the cache. |

Complete embeddings supply the inputs for fitting; they do not show whether prediction models have been trained or how accurately those models predict the benchmark outcome. The next check assembles each encoder triple, followed by the training and interpretation sections.

### Check each triple as training will read it

Open the original streams plus each replacement as a three-stream input. `BenchmarkStreams.open` checks row counts and codon-grid token lengths. The preceding preflight supplies the stronger recorded-input identity checks; token-length checks alone do not prove sequence identity. Rerunning this cell first releases the previous streams' mapped files.

Inspect the encoder names, grid resolutions and widths in the table before fitting. A cache mismatch stops preparation and needs to be resolved before interpreting a fusion comparison.

In [7]:
for previous in globals().get("STREAMS", {}).values():
    previous.close()
STREAMS = {}
for key in BENCHMARKS:
    for modality, encoder_key in swaps_for(key):
        STREAMS[key, encoder_key] = blf.BenchmarkStreams.open(key, CACHE_ROOT, {modality: encoder_key})
pd.DataFrame([{"benchmark": key, "swapped in": enc, "encoders": s.encoders,
               "grid": s.resolutions, "widths": s.widths}
              for (key, enc), s in STREAMS.items()]).set_index(["benchmark", "swapped in"])

encoders                                               grid                                  widths
benchmark           swapped in                                                                                                                                              
cov_vac_degradation nt500m      {'dna': 'nt500m', 'rna': 'rnafm', 'prot': 'esm...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...  {'dna': 1280, 'rna': 640, 'prot': 320}
                    hyenadna    {'dna': 'hyenadna', 'rna': 'rnafm', 'prot': 'e...  {'dna': 'nucleotide', 'rna': 'nucleotide', 'pr...   {'dna': 256, 'rna': 640, 'prot': 320}
                    rinalmo     {'dna': 'ntv2_100m', 'rna': 'rinalmo', 'prot':...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...   {'dna': 512, 'rna': 640, 'prot': 320}
                    mrnafm      {'dna': 'ntv2_100m', 'rna': 'mrnafm', 'prot': ...   {'dna': 'kmer', 'rna': 'codon', 'prot': 'codon'}  {'dna': 512, 'rna': 1280, 'prot': 320}
                    calm        {'dna': 'ntv2_100m', 'rna': 'calm', 'prot': 'e...   {'dna': 'kmer', 'rna': 'codon', 'prot': 'codon'}   {'dna': 512, 'rna': 768, 'prot': 320}
                    esm2_35m    {'dna': 'ntv2_100m', 'rna': 'rnafm', 'prot': '...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...   {'dna': 512, 'rna': 640, 'prot': 480}
                    esm2_150m   {'dna': 'ntv2_100m', 'rna': 'rnafm', 'prot': '...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...   {'dna': 512, 'rna': 640, 'prot': 640}
                    protbert    {'dna': 'ntv2_100m', 'rna': 'rnafm', 'prot': '...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...  {'dna': 512, 'rna': 640, 'prot': 1024}
fungal_expression   nt500m      {'dna': 'nt500m', 'rna': 'rnafm', 'prot': 'esm...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...  {'dna': 1280, 'rna': 640, 'prot': 320}
                    hyenadna    {'dna': 'hyenadna', 'rna': 'rnafm', 'prot': 'e...  {'dna': 'nucleotide', 'rna': 'nucleotide', 'pr...   {'dna': 256, 'rna': 640, 'prot': 320}
                    rinalmo     {'dna': 'ntv2_100m', 'rna': 'rinalmo', 'prot':...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...   {'dna': 512, 'rna': 640, 'prot': 320}
                    mrnafm      {'dna': 'ntv2_100m', 'rna': 'mrnafm', 'prot': ...   {'dna': 'kmer', 'rna': 'codon', 'prot': 'codon'}  {'dna': 512, 'rna': 1280, 'prot': 320}
                    calm        {'dna': 'ntv2_100m', 'rna': 'calm', 'prot': 'e...   {'dna': 'kmer', 'rna': 'codon', 'prot': 'codon'}   {'dna': 512, 'rna': 768, 'prot': 320}
                    esm2_35m    {'dna': 'ntv2_100m', 'rna': 'rnafm', 'prot': '...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...   {'dna': 512, 'rna': 640, 'prot': 480}
                    esm2_150m   {'dna': 'ntv2_100m', 'rna': 'rnafm', 'prot': '...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...   {'dna': 512, 'rna': 640, 'prot': 640}
                    protbert    {'dna': 'ntv2_100m', 'rna': 'rnafm', 'prot': '...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...  {'dna': 512, 'rna': 640, 'prot': 1024}
ecoli_proteins      nt500m      {'dna': 'nt500m', 'rna': 'rnafm', 'prot': 'esm...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...  {'dna': 1280, 'rna': 640, 'prot': 320}
                    hyenadna    {'dna': 'hyenadna', 'rna': 'rnafm', 'prot': 'e...  {'dna': 'nucleotide', 'rna': 'nucleotide', 'pr...   {'dna': 256, 'rna': 640, 'prot': 320}
                    rinalmo     {'dna': 'ntv2_100m', 'rna': 'rinalmo', 'prot':...  {'dna': 'kmer', 'rna': 'nucleotide', 'prot': '...   {'dna': 512, 'rna': 640, 'prot': 320}
                    mrnafm      {'dna': 'ntv2_100m', 'rna': 'mrnafm', 'prot': ...   {'dna': 'kmer', 'rna': 'codon', 'prot': 'codon'}  {'dna': 512, 'rna': 1280, 'prot': 320}
                    calm        {'dna': 'ntv2_100m', 'rna': 'calm', 'prot': 'e...   {'dna': 'kmer', 'rna': 'codon', 'prot': 'codon'}   {'dna': 512, 'rna': 768, 'prot': 320}
                    esm2_35m    {'dna': 'ntv2_100m', 'rna': 'rnafm', 'prot': '...  {'dna': 'km

---

## Part 2: Fit the encoder comparisons

### Run the baselines and fusion heads

**Why training follows extraction.** The pretrained encoders have converted sequences into numerical features. Training here keeps those embeddings fixed and learns the prediction head, together with any fusion and grid parameters used by that variant, to predict the benchmark's measured outcome. The pretrained encoders themselves are not fine-tuned.

The comparisons are the scientific purpose: does replacing one encoder improve the overall predictor, and does combining modalities add predictive value beyond the strongest individual encoder in that triple? For example, a replacement RNA encoder could improve both its single-encoder baseline and the fused model. Comparing fusion with the best single tests whether fusion adds predictive value beyond the strongest individual stream; comparing the fused model with the original triple measures the overall effect of the swap.

For each benchmark and replacement, fit the swapped-in encoder alone. When its stream is pooled or upsampled onto the grid, also fit the same encoder on that grid. Then compare concatenation, the gated MIL head and cross-attention using the replacement in its modality's slot.

**Execution order.** The first pass runs the unimodal baselines, concatenation and MIL for every selected pair; the second pass runs cross-attention. This prioritizes the cheaper comparisons, but an interruption before the first pass finishes can still leave some swaps unmeasured. No fixed overnight completion time is implied.

**Reuse and recovery.** Completed JSON results are reused when their benchmark, variant and full training settings match. This is not a complete check of data, cache and model-code identity. New epoch checkpoints retain optimizer, scheduler, random-generator and batch-order state. Resuming from the last saved epoch boundary can repeat up to ten epochs; older checkpoints warn when recovery state is missing. These mechanisms preserve work but do not promise identical numerical trajectories across hardware.

Keep the execution machine awake for an unattended run. On macOS, `caffeinate -i` can prevent idle sleep; use the corresponding power settings on Windows. The printed elapsed time applies to this invocation, including any reused results.

In [8]:
def config(**overrides):
    return train.TrainConfig(batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, channels=CHANNELS,
                             tconv=TCONV, seed=SEED, **overrides)


def swap_dir(key, encoder_key):
    return RESULTS_ROOT / key / encoder_key


def cheap_runs(key, modality, encoder_key):
    runs = [(f"unimodal:{modality}", {})]
    if STREAMS[key, encoder_key].resolutions[modality] != "codon":
        runs.append((f"unimodal-grid:{modality}", {}))
    return runs + [("concat", {}), ("mil", {"entropy_lambda": original_lambda(key)})]


started = time.perf_counter()
for key in BENCHMARKS:
    for modality, encoder_key in swaps_for(key):
        print(f"\n=== {key}: {ENCODERS[encoder_key].label} in the {modality} slot ===")
        for variant, overrides in cheap_runs(key, modality, encoder_key):
            train.run(STREAMS[key, encoder_key], variant, config(**overrides), DEVICE,
                      swap_dir(key, encoder_key))
for key in BENCHMARKS:
    for modality, encoder_key in swaps_for(key):
        print(f"\n=== {key}: {ENCODERS[encoder_key].label}, cross-attention ===")
        train.run(STREAMS[key, encoder_key], "cross", config(), DEVICE, swap_dir(key, encoder_key))
print(f"\nsweep finished in {(time.perf_counter() - started) / 3600:.2f} hours")


=== cov_vac_degradation: NT 500M human-ref in the dna slot ===
  cov_vac_degradation__unimodal-dna__lambda0.0__ch100__equation__seed0: already run, test 0.7269
  cov_vac_degradation__unimodal-grid-dna__lambda0.0__ch100__equation__seed0: already run, test 0.7318
  cov_vac_degradation__concat__lambda0.0__ch100__equation__seed0: already run, test 0.7531
  cov_vac_degradation__mil__lambda0.01__ch100__equation__seed0: already run, test 0.7103

=== cov_vac_degradation: HyenaDNA large in the dna slot ===
  cov_vac_degradation__unimodal-dna__lambda0.0__ch100__equation__seed0: already run, test 0.7548
  cov_vac_degradation__unimodal-grid-dna__lambda0.0__ch100__equation__seed0: already run, test 0.7733
  cov_vac_degradation__concat__lambda0.0__ch100__equation__seed0: already run, test 0.8319
  cov_vac_degradation__mil__lambda0.01__ch100__equation__seed0: already run, test 0.7974

=== cov_vac_degradation: RiNALMo 150M in the rna slot ===
  cov_vac_degradation__unimodal-rna__lambda0.0__ch100__equ

KeyboardInterrupt: 

---

## Part 3: Interpret the comparisons

### Compare each swap with its baselines

Each benchmark table starts with BioLangFusion's original triple, followed by the replacements. The single-encoder scores belong to the three encoders in that row's triple. Validation selects the “best single,” including a gridded reading where one is available.

**Two different questions.** Δfusion is a fusion head's test score minus its triple's validation-selected best single. “vs original” compares that fusion head with the same head using the original encoders on the same test rows. A replacement can improve the single encoder without increasing the benefit from fusion; read both differences together.

In [ ]:
SCALE = {"ecoli_proteins": 100.0}
HEADS = ("concat", "mil", "cross")


def swap_runs(key, encoder_key):
    return [train.Result.load(p) for p in sorted(swap_dir(key, encoder_key).glob("*.json"))]


def triple(key, modality=None, encoder_key=None):
    """{variant: Result} for one encoder triple: the original runs, with a swap's own runs
    replacing everything that involves the swapped slot."""
    lam = original_lambda(key)
    runs = {}
    for r in original_runs(key):
        name = label(r)
        if name == "mil" and r.config["entropy_lambda"] != lam:
            continue
        runs[name] = r
    if encoder_key is not None:
        for name in [n for n in runs if n in HEADS or n.endswith(f":{modality}")]:
            del runs[name]
        for r in swap_runs(key, encoder_key):
            runs[label(r)] = r
    return runs


def best_single(runs):
    singles = {n: r for n, r in runs.items() if n.startswith("unimodal") and r.val_score is not None}
    return max(singles, key=lambda n: singles[n].val_score) if singles else None


def summary(key):
    scale = SCALE.get(key, 1.0)
    original = triple(key)
    rows = []
    for modality, encoder_key in [(None, None), *swaps_for(key)]:
        runs = triple(key, modality, encoder_key)
        best = best_single(runs)
        row = {"slot": modality or "none",
               "encoder": ENCODERS[encoder_key].label if encoder_key else "BioLangFusion's own",
               "alone": runs.get(f"unimodal:{modality}").test_score * scale
               if modality and f"unimodal:{modality}" in runs else np.nan,
               "best single": best, "best single score": runs[best].test_score * scale if best else np.nan}
        for head in HEADS:
            score = runs[head].test_score * scale if head in runs else np.nan
            row[head] = score
            row[f"{head} Δfusion"] = score - row["best single score"]
            row[f"{head} vs original"] = (score - original[head].test_score * scale
                                          if head in original and encoder_key else np.nan)
        rows.append(row)
    return pd.DataFrame(rows).set_index(["slot", "encoder"])


for key in BENCHMARKS:
    print(f"\n{key}" + (" (accuracy x 100)" if key in SCALE else " (Spearman)"))
    try:
        display(summary(key).round(3))
    except (RuntimeError, KeyError) as missing:
        print("  not ready:", missing)

### Assess what the paired differences resolve

The paired bootstrap compares predictions on the same test rows. For each head and replacement, it estimates the difference from the original-encoder head and the difference from the replacement triple's best single. Exact duplicate sequences are grouped during resampling.

**Interpretation.** Read each interval against zero. An interval spanning zero leaves the direction unresolved under this procedure; it does not establish equivalence. Test-set sizes differ across the four benchmarks, so neither a universal row-count range nor a fixed 0.02 score threshold is justified. These intervals describe uncertainty from the evaluated rows, not variation across training seeds or protection against train/test sequence overlap.

In [ ]:
def resolve(key, draws=2000):
    streams = STREAMS[key, swaps_for(key)[0][1]]
    test = streams.split_rows("test")
    truth = streams.labels[test]
    groups = streams.rows["sequence"].to_numpy()[test]
    original = triple(key)
    predictions, comparisons = {}, []
    for name, r in original.items():
        predictions[f"original {name}"] = [np.asarray(r.test_prediction)]
    for modality, encoder_key in swaps_for(key):
        runs = triple(key, modality, encoder_key)
        best = best_single(runs)
        for name, r in runs.items():
            predictions[f"{encoder_key} {name}"] = [np.asarray(r.test_prediction)]
        for head in HEADS:
            if head not in runs or head not in original:
                continue
            comparisons.append((f"{encoder_key} {head} vs original",
                                f"{encoder_key} {head}", f"original {head}"))
            if best is not None:
                comparisons.append((f"{encoder_key} {head} vs best single ({best})",
                                    f"{encoder_key} {head}", f"{encoder_key} {best}"))
    out = pd.DataFrame(train.seed_averaged_bootstrap(streams.task, truth, predictions, comparisons,
                                                     groups=groups, draws=draws, seed=0))
    scale = SCALE.get(key, 1.0)
    for column in ("difference", "ci_low", "ci_high"):
        out[column] = out[column] * scale
    out["resolved"] = np.where((out["ci_low"] > 0) | (out["ci_high"] < 0), "yes", "no")
    return out.set_index("comparison")[["difference", "ci_low", "ci_high", "first_wins", "resolved"]]


RESOLVED = {}
for key in BENCHMARKS:
    try:
        RESOLVED[key] = resolve(key)
    except (RuntimeError, KeyError) as missing:
        print(f"{key}: not ready ({missing})")
        continue
    print(f"\n{key}")
    display(RESOLVED[key].round(3))

### Relate the comparisons to Stage 1 alignment

The Stage 2 decision is to report alignment beside fusion results. A replacement changes two of the triple's three encoder pairs. The table below gives Stage 1's raw and composition-controlled alignment for those pairs, using its 981-sequence mRNA stability sample, where all twelve encoders were measured together on coding sequences.

**Scope of the evidence.** These are measurements on that Stage 1 sample. They supply the frozen-encoder prediction being examined under H2; they are not newly measured alignment on each fusion benchmark.

**Open discrepancy.** The table reads the [Stage 1 capture](../../docs/figures/stability/results.json). Its raw CKA agrees with the saved Stage 1 notebook, but some composition-controlled values differ: RNA-FM with RiNALMo is 0.486 in the capture and 0.378 in the notebook. As recorded in the [run records](../../docs/run-records.md), that discrepancy remains unresolved. Treat the controlled values as provisional.

In [ ]:
stage1 = json.loads(STAGE1_ALIGNMENT.read_text())


def matrix(name):
    m = stage1["similarity"][name]
    return pd.DataFrame(m["data"], index=m["index"], columns=m["columns"])


cka, cka_controlled = matrix("Linear CKA"), matrix("Linear CKA after control")
alignment = []
for modality, encoder_key in sorted({s for b in BENCHMARKS for s in swaps_for(b)}):
    triple_keys = {**ORIGINAL, modality: encoder_key}
    for other in (m for m in ORIGINAL if m != modality):
        a, b = encoder_key, triple_keys[other]
        alignment.append({"swap": ENCODERS[encoder_key].label, "pair": f"{ENCODERS[a].label} + {ENCODERS[b].label}",
                          "CKA": cka.loc[a, b], "CKA after composition control": cka_controlled.loc[a, b]})
    alignment.append({"swap": ENCODERS[encoder_key].label,
                      "pair": f"{ENCODERS[encoder_key].label} vs the {ENCODERS[ORIGINAL[modality]].label} it replaces",
                      "CKA": cka.loc[encoder_key, ORIGINAL[modality]],
                      "CKA after composition control": cka_controlled.loc[encoder_key, ORIGINAL[modality]]})
pd.DataFrame(alignment).set_index(["swap", "pair"]).round(3)

### Draw conclusions from the available comparisons

Use the score differences, paired intervals and Stage 1 alignment together. The plan's success criterion is a reproducible improvement from at least one replacement under the unchanged architecture. A single-seed gain is a candidate for that conclusion, not sufficient evidence of reproducibility by itself.

A gain in the single-encoder baseline and a gain from fusion answer different questions. Likewise, an unresolved or negative swap result can reflect the original encoder's strength, the fixed head or limited transfer from the replacement. It does not by itself show that the replacement contains no useful biological information.

Two-encoder replacements are a later comparison named in the plan, chosen after the single swaps identify promising candidates. They are not executed by this notebook's current sweep.

## Choices and limitations

These choices define the present comparison in addition to Phase A's settings.

| Choice | Reason | Interpretation limit |
| --- | --- | --- |
| One seed per replacement | Keeps the initial sweep bounded | Bootstrap intervals do not measure training-seed variation; reproducibility needs further evidence |
| Reuse the original triple's gated-head entropy weight | Holds this tuning choice fixed across replacements | A replacement may behave differently at its own validation-selected weight |
| Leave DNABERT-2 outside the sweep | Variable-length byte-pair tokens need a fixed grid adapter | This DNA candidate remains untested here |
| Add RiNALMo and ESM-2 35M | Test replacements that Stage 1 found similar to the originals | Similarity motivates an H2 comparison; it does not guarantee little predictive change |
| Use alignment from the Stage 1 stability sample | Provides the existing twelve-encoder comparison | Values are sample-specific, and the controlled-CKA discrepancy remains open |
| Select three replacements for stability | Prioritize the mRFP-motivated RNA comparison, another codon encoder and a larger protein encoder | Five replacements remain untested there; older storage estimates are not verified requirements |
| Defer two-encoder replacements | Candidate triples depend on the single-swap evidence | This sweep does not complete the plan's later triple comparisons |